# Investigating top birder's behavior
I have to see whether I need to boot this manic birder out or not

### Investigating What eBird Checklists Actually Represent

I originally measured citizen-science activity by counting eBird checklist submissions across Chicago. However, after identifying one account responsible for 10,915 checklists in 2024, I wanted to understand what these reports actually contain.

I found that this observer submitted approximately 30 checklists per active day, with a median duration of just three minutes. Many reports were recorded at different locations around Edgewater, suggesting that one extended birdwatching outing might be divided into numerous short checklists.

I am now linking these checklists to the actual bird observation records to examine how many different bird taxa are reported per checklist, whether some checklists lack matching observations, and which birds appear most frequently.

My broader question is whether checklist volume accurately represents citizen-science observation effort, or whether individual reporting habits can significantly influence the geographic patterns in my data.

In [2]:
import pandas as pd
import altair as alt

observer = pd.read_csv("../data/observer_4476574_checklists.csv")
observer["OBSERVATION DATE"] = pd.to_datetime(observer["OBSERVATION DATE"])

observer.head()


,LAST EDITED DATE,COUNTRY,COUNTRY CODE,STATE,STATE CODE,COUNTY,COUNTY CODE,IBA CODE,BCR CODE,USFWS CODE,...,EFFORT AREA HA,NUMBER OBSERVERS,ALL SPECIES REPORTED,GROUP IDENTIFIER,CHECKLIST COMMENTS,Unnamed: 33,geometry,index_right,community,date
0,2025-09-18 19:55:46.658908,United States,US,Illinois,US-IL,Cook,US-IL-031,NaN,NaN,NaN,...,NaN,1.0,1,NaN,NaN,NaN,POINT (-87.6592684 42.0087282),0,ROGERS PARK,2023-10-25
1,2025-09-18 19:55:46.658908,United States,US,Illinois,US-IL,Cook,US-IL-031,NaN,NaN,NaN,...,NaN,1.0,1,NaN,NaN,NaN,POINT (-87.6592684 42.0087282),0,ROGERS PARK,2023-10-29
2,2025-09-18 19:55:46.658908,United States,US,Illinois,US-IL,Cook,US-IL-031,NaN,NaN,NaN,...,NaN,1.0,1,NaN,NaN,NaN,POINT (-87.6592684 42.0087282),0,ROGERS PARK,2023-11-11
3,2025-09-18 19:55:46.658908,United States,US,Illinois,US-IL,Cook,US-IL-031,NaN,NaN,NaN,...,NaN,1.0,1,NaN,NaN,NaN,POINT (-87.6592684 42.0087282),0,ROGERS PARK,2023-10-20
4,2025-09-18 19:55:46.658908,United States,US,Illinois,US-IL,Cook,US-IL-031,NaN,NaN,NaN,...,NaN,1.0,1,NaN,NaN,NaN,POINT (-87.6592684 42.0087282),0,ROGERS PARK,2023-11-22


In [6]:
observations_file = "../data/raw/ebird/cook_ebird_observations_2020_2025.txt"

# I check the available columns before loading the large dataset
columns = pd.read_csv(
    observations_file,
    sep="\t",
    nrows=0
).columns.tolist()

print(columns)

['GLOBAL UNIQUE IDENTIFIER', 'LAST EDITED DATE', 'TAXONOMIC ORDER', 'CATEGORY', 'TAXON CONCEPT ID', 'COMMON NAME', 'SCIENTIFIC NAME', 'SUBSPECIES COMMON NAME', 'SUBSPECIES SCIENTIFIC NAME', 'EXOTIC CODE', 'OBSERVATION COUNT', 'BREEDING CODE', 'BREEDING CATEGORY', 'BEHAVIOR CODE', 'AGE/SEX', 'COUNTRY', 'COUNTRY CODE', 'STATE', 'STATE CODE', 'COUNTY', 'COUNTY CODE', 'IBA CODE', 'BCR CODE', 'USFWS CODE', 'ATLAS BLOCK', 'LOCALITY', 'LOCALITY ID', 'LOCALITY TYPE', 'LATITUDE', 'LONGITUDE', 'OBSERVATION DATE', 'TIME OBSERVATIONS STARTED', 'OBSERVER ID', 'OBSERVER ORCID ID', 'SAMPLING EVENT IDENTIFIER', 'OBSERVATION TYPE', 'PROTOCOL NAME', 'PROTOCOL CODE', 'PROJECT NAMES', 'PROJECT IDENTIFIERS', 'DURATION MINUTES', 'EFFORT DISTANCE KM', 'EFFORT AREA HA', 'NUMBER OBSERVERS', 'ALL SPECIES REPORTED', 'GROUP IDENTIFIER', 'HAS MEDIA', 'APPROVED', 'REVIEWED', 'REASON', 'CHECKLIST COMMENTS', 'SPECIES COMMENTS', 'Unnamed: 52']


In [8]:
# load the checklist records for my most active observer
observer = pd.read_csv("../data/observer_4476574_checklists.csv")

# I identify the checklists that I want to investigate
checklist_ids = set(observer["SAMPLING EVENT IDENTIFIER"])

print("Observer checklists:", len(checklist_ids))

Observer checklists: 12405


In [9]:
# select only the observation fields I need
cols = [
    "SAMPLING EVENT IDENTIFIER",
    "COMMON NAME",
    "SCIENTIFIC NAME",
    "OBSERVATION COUNT"
]

# I read the large observations file in smaller pieces
matches = []

for chunk in pd.read_csv(
    observations_file,
    sep="\t",
    usecols=cols,
    chunksize=200_000
):
    subset = chunk[
        chunk["SAMPLING EVENT IDENTIFIER"].isin(checklist_ids)
    ]

    if not subset.empty:
        matches.append(subset)

observations = pd.concat(matches, ignore_index=True) if matches else pd.DataFrame(columns=cols)

print("Bird observation records:", len(observations))
print("Checklists with bird observations:",
      observations["SAMPLING EVENT IDENTIFIER"].nunique())

Bird observation records: 51307
Checklists with bird observations: 12197


In [ ]:
# count the distinct reported taxa associated with each checklist
species_counts = (
    observations.groupby("SAMPLING EVENT IDENTIFIER")
    ["SCIENTIFIC NAME"]
    .nunique()
    .reset_index(name="species_count")
)

# species counts to my observer's checklists
observer_species = observer.merge(
    species_counts,
    on="SAMPLING EVENT IDENTIFIER",
    how="left"
)

print("Reported taxa per matched checklist:")
display(observer_species["species_count"].describe())

print("Checklists without matching observations:")
print(observer_species["species_count"].isna().sum())

print("\nMost frequently reported birds:")
display(observations["COMMON NAME"].value_counts().head(20))

Reported taxa per matched checklist:


count    12197.000000
mean         4.206526
std          3.583708
min          1.000000
25%          2.000000
50%          3.000000
75%          5.000000
max         40.000000
Name: species_count, dtype: float64

Checklists without matching observations:
208

Most frequently reported birds:


COMMON NAME
House Sparrow             11657
Rock Pigeon                6096
American Robin             4666
European Starling          4455
Northern Cardinal          3231
House Finch                2390
Ring-billed Gull           1769
White-throated Sparrow     1523
Black-capped Chickadee     1390
Chimney Swift              1384
Mourning Dove              1204
Downy Woodpecker            980
Golden-crowned Kinglet      906
Dark-eyed Junco             824
American Goldfinch          728
Canada Goose                683
American Crow               669
Common Grackle              558
Red-winged Blackbird        543
Barn Swallow                427
Name: count, dtype: int64